In [ ]:
#------------------------------------------------------------------------------------------

import numpy as np
import open3d as o3d

points_camera = np.array([
    [84.3891, 13.2412, 166.6440],
    [83.9518, -0.1961, 159.9560],
    [83.9536, -14.9789, 160.3110],
    [84.0755, -27.9511, 167.1670],
    
    [69.8635, 13.0988, 167.2780],
    [69.1641, -0.1976, 161.1560],
    [69.0972, -14.8977, 161.4330],
    [69.6256, -27.7197, 168.1000],
    
    [55.0997, 13.2110, 168.7110],
    [54.8202, -0.3857, 162.1330],
    [54.7456, -14.8058, 162.4670],
    [54.9811, -27.6642, 168.9440],
    
    [40.4586, 13.0611, 169.2890],
    [40.3147, -0.3882, 163.1780],
    [40.5525, -15.0772, 163.3780],
    [40.6552, -27.6591, 170.1110]
])

points_end = np.array([
    [-24.2542, 20.4890, 127.2864],
    [-30.7397, 7.9457, 127.1185],
    [-30.9790, -6.9546, 126.8909],
    [-24.2057, -20.5462, 127.1830],
    [-24.2150, 20.5352, 141.7383],
    [-30.7860, 7.7641, 141.5236],
    [-30.7094, -8.0620, 141.5009],
    [-24.0816, -20.6916, 141.6253],
    [-24.4506, 20.2541, 154.6282],
    [-30.8851, 7.3601, 154.2075],
    [-30.4154, -9.1086, 154.4154],
    [-23.8859, -20.9172, 154.6171],
    [-24.7038, 19.9446, 170.7040],
    [-31.1840, 5.9683, 170.3836],
    [-30.6528, -8.2746, 170.3755],
    [-24.2139, -20.5366, 170.7988]
])


# Step 1: Coarse registration
# Compute centroids and center the points
centroid_camera = np.mean(points_camera, axis=0)
centroid_end = np.mean(points_end, axis=0)

points_camera_centered = points_camera - centroid_camera
points_end_centered = points_end - centroid_end

# Compute the covariance matrix
H = np.dot(points_camera_centered.T, points_end_centered)
U, S, Vt = np.linalg.svd(H)

R_init = np.dot(Vt.T, U.T)
if np.linalg.det(R_init) < 0:
    Vt[-1, :] *= -1
    R_init = np.dot(Vt.T, U.T)

t_init = centroid_end - np.dot(R_init, centroid_camera)

# Initial transformation matrix
T_init = np.eye(4)
T_init[:3, :3] = R_init
T_init[:3, 3] = t_init

# Apply the initial transformation to the point cloud
points_camera_transformed = (R_init @ points_camera.T).T + t_init

# Step 2: Fine registration using ICP
# Create Open3D point cloud objects
pcd_camera = o3d.geometry.PointCloud()
pcd_camera.points = o3d.utility.Vector3dVector(points_camera_transformed)

pcd_end = o3d.geometry.PointCloud()
pcd_end.points = o3d.utility.Vector3dVector(points_end)

# Run ICP
threshold = 5  # maximum correspondence distance
icp_result = o3d.pipelines.registration.registration_icp(
    pcd_camera, pcd_end, threshold,
    np.eye(4),  # initial transformation matrix
    o3d.pipelines.registration.TransformationEstimationPointToPoint()
)

# Output the fine-registration result
T_icp = icp_result.transformation

# Combine the coarse transformation with the ICP result
T_final = T_icp @ T_init

# Print results
print("Initial transformation matrix (coarse registration):\n", T_init)
print("ICP transformation matrix (fine registration):\n", T_icp)
print("Final transformation matrix T_final:\n", T_final)

# Visualize the transformed result
pcd_camera_transformed_icp = pcd_camera.transform(T_icp)

o3d.visualization.draw_geometries([pcd_camera_transformed_icp, pcd_end],
                                  window_name="ICP fine registration result",
                                  point_show_normal=False,
                                  width=800,
                                  height=600)